# 為什麼使用 Discord Bot？

## PyLadies Kaohsiung 選擇 Discord 的原因
- 線上直播
- 社群成員交流
- 權限控管功能強大
- 訊息無時效限制

## Bot 可以做什麼？


- 社群管理與互動
    - 自動審核
    - 權限處理
    - 移除不適當內容
    - 抽獎功能
    - 成長等級系統
- 個人/團隊工作
    - 定時排程
    - Agent 串接
    - 訂單、客服系統
    - 最新訊息追蹤 (Trello, Youtube, 網站)

> - [MEE6](https://mee6.xyz/)
> - [nekoHaru](https://harutoo.com/)
> - [Kiro Discord Bot](https://gist.github.com/nczz/de67be1fc2effafd2013ed804650dc73#91-%E9%A0%BB%E9%81%93%E5%B0%B1%E6%98%AF%E5%B7%A5%E4%BD%9C%E5%8D%80%E7%94%A8%E9%A0%BB%E9%81%93%E4%BE%86%E5%88%86%E5%B7%A5)

# 環境設定

In [ ]:
!pip install "discord.py>=2.3.0"

In [ ]:
# colab 才需要執行
import nest_asyncio
nest_asyncio.apply()

In [ ]:
GUILD_ID = 伺服器ID # 步驟 1.3
BOT_TOKEN = ""     # 步驟 2.4

# Discord Bot 基礎

## 指令

In [ ]:
import discord
from discord import app_commands

intents = discord.Intents.default()
client = discord.Client(intents=intents)
tree = app_commands.CommandTree(client)

### 指令 ====================================
@tree.command(name="hi", description="跟 bot 打招呼")
async def hi(interaction: discord.Interaction):
    msg = "Hello World"
    await interaction.response.send_message(msg)

### 啟動 bot ====================================
@client.event
async def on_ready():
    if GUILD_ID:
        guild = discord.Object(id=GUILD_ID)
        tree.copy_global_to(guild=guild)
        await tree.sync(guild=guild)
    else:
        await tree.sync()
    print(f"已登入為 {client.user}，指令同步完成")


client.run(BOT_TOKEN)

這時回到 Discord
輸入指令 `/hi` 就可以看到機器人的回覆

### [練習] 修改指令與回覆內容
- 試著把指令 `/hi` 改成 `/hello`, 回覆語句改成 `f"Hi {interaction.user.mention}"`

## 共用程式碼

剛剛那一格裡，「建立 bot」和「啟動 bot」這兩段之後每次都一樣。  
先把它們包成函式，後面就只要專心寫**新增的功能**。

In [ ]:
### 共用程式碼 ====================================
import discord
from discord import app_commands


def new_bot():
    """建立一隻全新的 bot，回傳 (client, tree)。

    每一格都重新建一隻，同一格重複執行也不會出現「指令已經註冊過」的錯誤。
    """
    intents = discord.Intents.default()
    client = discord.Client(intents=intents)
    tree = app_commands.CommandTree(client)
    return client, tree


def run_bot(client, tree, views=()):
    """啟動 bot：註冊按鈕、同步指令，然後開始運作。

    views：要重新註冊的按鈕，讓「之前發布過」的按鈕在 bot 重啟後依然可以按。
    """

    @client.event
    async def on_ready():
        for view in views:
            client.add_view(view)

        if GUILD_ID:
            guild = discord.Object(id=int(GUILD_ID))
            tree.copy_global_to(guild=guild)
            await tree.sync(guild=guild)
        else:
            await tree.sync()
        print(f"已登入為 {client.user}，指令同步完成")

    client.run(BOT_TOKEN)

## 按鈕

到目前為止，每次要跟 bot 互動都得自己打 `/hi`。  
指令很好寫，但對使用者來說門檻不低，  
要記得指令叫什麼、不能打錯字，手機上打字更麻煩。

**按鈕**可以解決這件事。
雖然第一次還是要用指令發布一顆可以點選的按鈕，  
但那則訊息會一直留在頻道裡，之後所有人都用點的就好，不必再輸入任何東西。

In [ ]:
import discord

### 按鈕 ====================================
# 指令只能回文字；要做出可以點的按鈕，得先用 discord.ui.View 把按鈕包起來，
# 再讓指令把這個 View 跟訊息一起送出去。
class HiView(discord.ui.View):
    """按鈕本體。timeout=None + 固定 custom_id，讓按鈕永久有效、重啟也能用。"""

    def __init__(self):
        super().__init__(timeout=None)

    @discord.ui.button(
        label="跟 bot 打招呼",
        style=discord.ButtonStyle.primary,
        custom_id="hi_button",
    )
    async def hi_button(
        self, interaction: discord.Interaction, button: discord.ui.Button
    ):
        msg = "Hello World"
        await interaction.response.send_message(msg)


### 指令 ====================================
client, tree = new_bot()

# 指令負責「把按鈕發出去」，view=HiView() 就是把按鈕掛在這則訊息上
@tree.command(name="hi-button", description="發布一顆可以跟 bot 打招呼的按鈕")
async def hi_button_command(interaction: discord.Interaction):
    msg = "點下面的按鈕跟我打招呼"
    await interaction.response.send_message(msg, view=HiView())


# views=[HiView()] 會在啟動時重新註冊按鈕，讓之前發出去的按鈕重啟後還能按
run_bot(client, tree, views=[HiView()])

### [練習] [午餐機器人](https://python-tutorial-workshop.github.io/#curriculum)
- 複製前面的範例程式碼，  
把以下程式碼的 `msg = f"推薦你可以吃{get_random_meal()}"` 取代原本範例中的 `msg = "Hello World"` 那行
- 改[按鈕風格](https://discordpy.readthedocs.io/en/stable/interactions/api.html#discord.ButtonStyle)

### [進階練習]
可以嘗試重新命名
- 顯示
    - 指令說明
    - 按鈕文字
- 程式碼：
  - 指令名稱：`hi-button` -> `pick-meal`
  - 指令函式名稱：`hi_button_command` -> `pick_meal_command`
  - View 名稱：`HiView` -> `MealView`
  - 按鈕函式名稱：`hi_button` -> `get_random_meal_button`



In [ ]:
import random

meal_choices = ["漢堡王", "八方雲集", "7-11", "すき家"]

def get_random_meal():
    return random.choice(meal_choices)

msg = f"推薦你可以吃 {get_random_meal()}" ### 複製這行取代範例中的 msg = "Hello World" 那行
print(msg)

In [ ]:
### 請將程式碼複製到此處，並作修改




## 增加/列出餐食選項

In [ ]:
import random
import discord

meal_choices = ["漢堡王", "八方雲集", "7-11", "すき家"]

def get_random_meal():
    return random.choice(meal_choices)


### 按鈕 ====================================
class MealView(discord.ui.View):
    def __init__(self):
        super().__init__(timeout=None)

    @discord.ui.button(
        label="抽一個吃什麼",
        style=discord.ButtonStyle.primary,
        custom_id="get_random_meal_button",
    )
    async def get_random_meal_button(
        self, interaction: discord.Interaction, button: discord.ui.Button
    ):
        msg = f"推薦你可以吃 {get_random_meal()}"
        await interaction.response.send_message(msg)


### 指令 ====================================
client, tree = new_bot()


@tree.command(name="pick-meal", description="發布一顆抽選項的按鈕")
async def pick_meal_command(interaction: discord.Interaction):
    msg = "點下面的按鈕抽一個選項"
    await interaction.response.send_message(msg, view=MealView())

### 此次新增 start -----------------------------------------。
@tree.command(name="add-meal", description="新增一個餐食選項")
@app_commands.describe(food="想加入選項的店名或食物")
async def add_meal_command(interaction: discord.Interaction, food: str):
    meal_choices.append(food)
    msg = f"已新增「{food}」，目前選項有：{', '.join(meal_choices)}"
    await interaction.response.send_message(msg)


@tree.command(name="list-meals", description="查看目前有哪些餐食選項")
async def list_meals_command(interaction: discord.Interaction):
    msg = f"目前選項有：{', '.join(meal_choices)}"
    await interaction.response.send_message(msg)
### 此次新增 end -----------------------------------------


run_bot(client, tree, views=[MealView()])

試試看在增加餐食後，把 bot 停掉，重新執行一次，  
然後在 Discord 輸入 `/list-meals`。  
會發現剛剛用 `/add-meal` 加進去的選項全都不見了。

原因是 `meal_choices` 只是一個 Python 的 list，  
它活在程式的記憶體裡，程式一結束就跟著消失。

# 資料庫串接

要讓資料在重啟之後還在，就需要一個**可以長期儲存資料的地方**，也就是**資料庫**。  
這次我們用 **SQLite**：它不需要另外架伺服器，整個資料庫就是一個檔案（`db.db`），  
Python 內建就能操作，很適合這種小型專案。

> 在 Colab 上 `db.db` 是存在執行階段（runtime）裡的，  
> 重跑 cell 沒問題，但整個 runtime 重置就會不見。  
> 在自己電腦上執行時，它就是一個真正留在硬碟上的檔案了。

更多資料庫的說明與介紹，可以參考之前的[工作坊講義](https://python-practical-workshop.github.io/sql_flask/)。


## 增加/列出餐食選項

In [ ]:
### 資料庫 ====================================

import sqlite3

DB_PATH = "db.db"


def init_db():
    """建立資料表（如果還不存在），並在資料表是空的時候塞入預設選項。"""
    conn = sqlite3.connect(DB_PATH)
    conn.execute(
        """
        CREATE TABLE IF NOT EXISTS meals (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL
        )
        """
    )
    (count,) = conn.execute("SELECT COUNT(*) FROM meals").fetchone()
    if count == 0:
        default_meals = ["漢堡王", "八方雲集", "7-11", "すき家"]
        conn.executemany(
            "INSERT INTO meals (name) VALUES (?)",
            [(name,) for name in default_meals],
        )
    conn.commit()
    conn.close()


def get_all_meals():
    """從資料庫撈出所有選項，回傳一個 list。"""
    conn = sqlite3.connect(DB_PATH)
    rows = conn.execute("SELECT name FROM meals").fetchall()
    conn.close()
    return [row[0] for row in rows]


def add_meal(food):
    """把一個新選項寫進資料庫。"""
    conn = sqlite3.connect(DB_PATH)
    conn.execute("INSERT INTO meals (name) VALUES (?)", (food,))
    conn.commit()
    conn.close()


def format_meal_list():
    """把目前所有選項排成一行字，例如「漢堡王, 八方雲集, 7-11」。

    後面好幾個地方都要顯示選項清單，統一寫在這裡，
    之後想改成換行顯示或加編號，只要改這一個函式。
    """
    return ", ".join(get_all_meals())


In [ ]:
import random

### 此次調整 start -----------------------------------------
def get_random_meal():
    """用 random.choice() 從資料庫的選項中抽一筆，回傳名稱字串。"""
    meal_choices = get_all_meals()
    if not meal_choices:
        return "資料庫裡還沒有任何選項"
    return random.choice(meal_choices)
### 此次調整 end -------------------------------------------

# MealView 不用重寫：它呼叫的是 get_random_meal()，
# 上面重新定義之後，按鈕自動就會用到新版本

client, tree = new_bot()

### 指令(每次都需註冊) ====================================
@tree.command(name="pick-meal", description="發布一顆抽選項的按鈕")
async def pick_meal_command(interaction: discord.Interaction):
    msg = "點下面的按鈕抽一個選項"
    await interaction.response.send_message(msg, view=MealView())


### 此次調整 start -----------------------------------------
@tree.command(name="add-meal", description="新增一個餐食選項")
@app_commands.describe(food="想加入選項的店名或食物")
async def add_meal_command(interaction: discord.Interaction, food: str):
    add_meal(food)
    msg = f"已新增「{food}」，目前選項有：{format_meal_list()}"
    await interaction.response.send_message(msg)


@tree.command(name="list-meals", description="查看目前有哪些餐食選項")
async def list_meals_command(interaction: discord.Interaction):
    msg = f"目前選項有：{format_meal_list()}"
    await interaction.response.send_message(msg)


init_db()
### 此次調整 end -------------------------------------------
run_bot(client, tree, views=[MealView()])


### [練習] 更改預設餐食
- 請刪掉目前的資料庫檔案 `db.db`
- 更改程式碼預設的餐食選項

## 刪除餐食選項

In [ ]:
### 此次新增
def get_all_meals_with_id():
    """從資料庫撈出所有選項，回傳 [(id, name), ...] 的 list。
    刪除功能要用 id 對應，避免同名選項刪錯筆。"""
    conn = sqlite3.connect(DB_PATH)
    rows = conn.execute("SELECT id, name FROM meals").fetchall()
    conn.close()
    return rows


def delete_meal(meal_id):
    """依照 id 刪除一筆選項。"""
    conn = sqlite3.connect(DB_PATH)
    conn.execute("DELETE FROM meals WHERE id = ?", (meal_id,))
    conn.commit()
    conn.close()

In [ ]:
### 此次新增 start -----------------------------------------
class DeleteMealSelect(discord.ui.Select):
    def __init__(self):
        # Select 選單最多只能放 25 個選項，資料庫筆數超過的話這裡先截斷
        options = [
            discord.SelectOption(label=name, value=str(meal_id))
            for meal_id, name in get_all_meals_with_id()[:25]
        ]
        super().__init__(
            placeholder="選擇要刪除的選項",
            options=options,
        )

    async def callback(self, interaction: discord.Interaction):
        meal_id = int(self.values[0])
        delete_meal(meal_id)

        msg = f"已刪除，目前選項有：{format_meal_list()}"
        # 用 edit_message 原地覆蓋掉「帶選單的那則訊息」，
        # 不要另外 send_message 發新訊息，否則舊的選單會一直殘留在頻道裡
        await interaction.response.edit_message(content=msg, view=None)


class DeleteMealView(discord.ui.View):
    def __init__(self):
        super().__init__(timeout=None)
        self.add_item(DeleteMealSelect())
### 此次新增 end -------------------------------------------

client, tree = new_bot()



### 指令(每次都需註冊) ====================================
@tree.command(name="pick-meal", description="發布一顆抽選項的按鈕")
async def pick_meal_command(interaction: discord.Interaction):
    msg = "點下面的按鈕抽一個選項"
    await interaction.response.send_message(msg, view=MealView())


@tree.command(name="add-meal", description="新增一個餐食選項")
@app_commands.describe(food="想加入選項的店名或食物")
async def add_meal_command(interaction: discord.Interaction, food: str):
    add_meal(food)
    msg = f"已新增「{food}」，目前選項有：{format_meal_list()}"
    await interaction.response.send_message(msg)


@tree.command(name="list-meals", description="查看目前有哪些餐食選項")
async def list_meals_command(interaction: discord.Interaction):
    msg = f"目前選項有：{format_meal_list()}"
    await interaction.response.send_message(msg)


### 此次新增 start -----------------------------------------
@tree.command(name="delete-meal", description="用選單刪除一個餐食選項")
async def delete_meal_command(interaction: discord.Interaction):
    if not get_all_meals_with_id():
        await interaction.response.send_message("目前沒有任何選項可以刪除")
        return

    await interaction.response.send_message(
        "選一個要刪除的選項", view=DeleteMealView()
    )
### 此次新增 end -------------------------------------------

init_db()
run_bot(client, tree, views=[MealView()])


### [練習] 刪除所有餐食選項
增加一個指令 `/delete-all-meals`, 可以刪除所有既有的餐食選項
- 想一想應該要參考哪一個指令的程式碼？需不需要輸入內容？需不需要選單？
- 資料庫的操作可以參考 `delete_meal`, 但這次是要刪全部

```python
conn.execute("DELETE FROM meals")
```

# Discord Bot 進階

## 整合面板：多按鈕、文字輸入

現在管理選項的操作有三個指令：  
`/add-meal`、`/list-meals`、`/delete-meal`。  
功能很方便，但使用者得記住三個指令名稱才用得起來。

前面提過按鈕在互動上更方便，所以接下來我們做一個**整合面板**：  
一則訊息裡放多顆按鈕，點哪一顆就做哪件事，跟三個指令做的完全是同一件事。  

在新增餐食的部分，因為 **按鈕收不到文字**，  
我們會多用到 **Modal**，也就是點下去之後跳出來的那種輸入表單。  

In [ ]:
### 此次新增 start -----------------------------------------
# 按鈕收不到文字。要讓使用者打字，得跳出一個 Modal 表單。
class AddMealModal(discord.ui.Modal, title="新增選項"):
    food = discord.ui.TextInput(label="想加入的店名或食物", max_length=50)

    async def on_submit(self, interaction: discord.Interaction):
        add_meal(self.food.value)
        msg = f"已新增「{self.food.value}」，目前選項有：{format_meal_list()}"
        await interaction.response.send_message(msg)


# 一個 View 其實可以放好幾顆按鈕，
# 把「列出／新增／刪除」三個功能收在同一則訊息裡。
class MealManagerView(discord.ui.View):
    def __init__(self):
        super().__init__(timeout=None)

    @discord.ui.button(
        label="列出現有選項",
        style=discord.ButtonStyle.secondary,
        custom_id="manage_list_button",
    )
    async def list_button(
        self, interaction: discord.Interaction, button: discord.ui.Button
    ):
        msg = f"目前選項有：{format_meal_list()}"
        await interaction.response.send_message(msg)

    @discord.ui.button(
        label="新增選項",
        style=discord.ButtonStyle.success,
        custom_id="manage_add_button",
    )
    async def add_button(
        self, interaction: discord.Interaction, button: discord.ui.Button
    ):
        # 按鈕收不到文字，要靠 Modal 跳出表單讓使用者輸入
        await interaction.response.send_modal(AddMealModal())

    @discord.ui.button(
        label="刪除選項",
        style=discord.ButtonStyle.danger,
        custom_id="manage_delete_button",
    )
    async def delete_button(
        self, interaction: discord.Interaction, button: discord.ui.Button
    ):
        if not get_all_meals_with_id():
            await interaction.response.send_message(
                "目前沒有任何選項可以刪除"
            )
            return
        # 沿用上一格做好的 DeleteMealView（動態組出目前的選單）
        await interaction.response.send_message(
            "選一個要刪除的選項", view=DeleteMealView()
        )
### 此次新增 end -------------------------------------------


client, tree = new_bot()

### 指令(每次都需註冊) ====================================
@tree.command(name="pick-meal", description="發布一顆抽選項的按鈕")
async def pick_meal_command(interaction: discord.Interaction):
    msg = "點下面的按鈕抽一個選項"
    await interaction.response.send_message(msg, view=MealView())


### 此次調整 start -----------------------------------------
# 移除 /add-meal , /list-meals

@tree.command(name="manage-meals", description="發布一個可以列出／新增／刪除選項的面板")
async def manage_meals_command(interaction: discord.Interaction):
    await interaction.response.send_message("選項管理面板", view=MealManagerView())
### 此次調整 end -------------------------------------------

init_db()

### 此次調整 start -----------------------------------------
run_bot(client, tree, views=[MealView(), MealManagerView()])
### 此次調整 end -------------------------------------------


### [練習] 整合
- 新增一個按鈕，把剛才的刪除所有餐食選項的功能加入

## 訊息發送方式

- 建立一個新的文字頻道
- 複製頻道ID，貼到以下程式碼中

In [ ]:
import discord
from discord import app_commands

TARGET_CHANNEL_ID = 新頻道ID


client, tree = new_bot()

### 指令與按鈕 ====================================
# /hi-private：只有按的人看得到
class PrivateHiView(discord.ui.View):
    def __init__(self):
        super().__init__(timeout=None)

    @discord.ui.button(
        label="跟 bot 打招呼",
        style=discord.ButtonStyle.primary,
        custom_id="hi_private_button",
    )
    async def hi_button(
        self, interaction: discord.Interaction, button: discord.ui.Button
    ):
        msg = "Hello World"
        await interaction.response.send_message(msg, ephemeral=True)


@tree.command(name="hi-private", description="跟 bot 打招呼，只有按的人看得到回覆")
async def hi_private(interaction: discord.Interaction):
    await interaction.response.send_message(
        "點下面的按鈕跟我打招呼", view=PrivateHiView()
    )


# /hi-dm：私訊給按的人
class DMHiView(discord.ui.View):
    def __init__(self):
        super().__init__(timeout=None)

    @discord.ui.button(
        label="跟 bot 打招呼",
        style=discord.ButtonStyle.primary,
        custom_id="hi_dm_button",
    )
    async def hi_button(
        self, interaction: discord.Interaction, button: discord.ui.Button
    ):
        try:
            msg = "Hello World"
            await interaction.user.send(msg)
            await interaction.response.send_message(
                "已經私訊給你囉，記得檢查一下", ephemeral=True
            )
        except discord.Forbidden:
            # 對方關閉了「允許來自伺服器成員的私訊」
            await interaction.response.send_message(
                "私訊失敗了，請先開啟允許接收私訊的設定", ephemeral=True
            )


@tree.command(name="hi-dm", description="跟 bot 打招呼，只有按的人會收到私訊回覆")
async def hi_dm(interaction: discord.Interaction):
    await interaction.response.send_message(
        "點下面的按鈕跟我打招呼", view=DMHiView()
    )

# /hi-channel：回覆改發到另一個頻道
class ChannelHiView(discord.ui.View):
    def __init__(self):
        super().__init__(timeout=None)

    @discord.ui.button(
        label="跟 bot 打招呼",
        style=discord.ButtonStyle.primary,
        custom_id="hi_channel_button",
    )
    async def hi_button(
        self, interaction: discord.Interaction, button: discord.ui.Button
    ):
        msg = "Hello World"
        if not TARGET_CHANNEL_ID:
            await interaction.response.send_message(
                "尚未設定 DISCORD_TARGET_CHANNEL_ID，無法發送", ephemeral=True
            )
            return

        # 先確認互動，避免 3 秒內沒回應顯示「互動失敗」
        await interaction.response.send_message(
            "已經幫你發到指定頻道囉", ephemeral=True
        )

        target = client.get_channel(
            int(TARGET_CHANNEL_ID)
        ) or await client.fetch_channel(int(TARGET_CHANNEL_ID))
        await target.send(f"{msg}: {interaction.user.mention}")


@tree.command(name="hi-channel", description="發布按鈕，回覆會發到另一個頻道")
async def hi_channel(interaction: discord.Interaction):
    await interaction.response.send_message(
        "點下面的按鈕跟我打招呼", view=ChannelHiView()
    )

run_bot(client, tree, views=[PrivateHiView(), DMHiView(), ChannelHiView()])


# 延伸學習

Discord bot 能做的事遠遠不只今天這些，  
可以參考 [discord.py 官方文件](https://discordpy.readthedocs.io/en/stable/index.html)，
今天只用到其中很小一部分，  
文件裡還有語音頻道、身分組管理、訊息監聽、排程任務、  
以及各種 UI 元件（今天用了 Button、Select、Modal，其實還有更多）。